0. 这次要求大家编写一个鱼C超市会员管理系统。        
程序功能要求如下：      
1. 会员开卡     
会员开卡自动赋予一个新的卡号（默认从 10000 开始递增）       
要求输入用户名和密码        
密码要求长度不低于 6 位     
2. 修改密码     
先确认输入卡号是否存在      
再判断旧密码是否正确        
新密码长度同样不能低于 6 位     
3. 商品支付     
先确认输入卡号是否存在
再判断密码是否正确
根据输入的消费金额增加会员卡积分（比如输入 250 消费金额，那么积分增加 250）     
4. 积分查询     
先确认输入卡号是否存在      
再判断密码是否正确      
代码要求：      
1. 创建一个会员类（Member），包含信息：卡号、用户名、密码（要求密码需要以 MD5 哈希值的形式存储）、积分、注册日期        
2. 创建一个管理类（Manage），用于实现上方要求的主要程序功能     
3. 创建一个 Mixin 类（PasswdMixin），用于实现密码相关的功能组件（密码长度检测和将明文密码转换为 MD5 的操作）        
4. 创建一个 Mixin 类（LoggerMixin），用于实现日志记录相关的功能组件

In [ ]:
import time
import hashlib
card_num = 10000

class PasswdMixin:
    def check_password_length(self, password):
        return len(password) >= 6
    
    def hash_password(self, password):
        md5 = hashlib.md5()
        md5.update(password.encode('utf-8'))
        return md5.hexdigest()

class LoggerMixin:
    def log(self, message):
        with open('log.txt', 'a', encoding='utf-8') as f:
            timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
            f.write(f'{message}，时间：{timestamp}\n')

class Member:
    def __init__(self, card_num):
        self.username = ""
        self.password = ""
        self.card_num = card_num
        self.integral = 0
        self.register_date = ""

class Manage(PasswdMixin, LoggerMixin):
    def __init__(self):
        self.members = []
    
    def new_card(self):
        global card_num
        username = input('请输入名字：')
        while True:
            password = input('请输入密码：')
            if not self.check_password_length(password):
                print('会员密码不能小于6位，请重新输入：')
                continue
            break
        member = Member(card_num)
        card_num += 1
        member.username = username
        member.password = self.hash_password(password)
        member.register_date = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
        self.members.append(member)
        print(f'创建成功，卡号为 {card_num-1}，关联用户 -> {username}')
        self.log(f'开卡成功：{card_num-1} -> {username}')
    
    def modify_password(self):
        card_num = int(input('请输入卡号：'))
        member = None
        for m in self.members:
            if m.card_num == card_num:
                member = m
                break
        if not member:
            print('会员不存在！')
            return
        password = input('请输入密码：')
        if self.hash_password(password) != member.password:
            print('密码错误！')
            return
        while True:
            new_password = input('请输入新密码：')
            if not self.check_password_length(new_password):
                print('会员密码不能小于6位，请重新输入：')
                continue
            break
        member.passport = self.hash_password(new_password)
        print('密码修改成功。')
        self.log(f'修改密码：卡号 -> {card_num}')

    def pay(self):
        card_num = int(input('请输入卡号：'))
        member = None
        for m in self.members:
            if m.card_num == card_num:
                member = m
                break
        if not member:
            print('会员不存在！')
            return
        password = input('请输入密码：')
        if self.hash_password(password) != member.password:
            print('密码错误！')
            return
        amount = int(input('请输入支付金额：'))
        member.integral += amount
        print(f'支付成功，增加 {amount} 积分')
        self.log(f'积分累计：卡号 -> {card_num}，+{amount}分')

    def inquiry_integral(self):
        card_num = int(input('请输入卡号：'))
        member = None
        for m in self.members:
            if m.card_num == card_num:
                member = m
                break
        if not member:
            print('会员不存在！')
            return
        password = input('请输入密码：')
        if self.hash_password(password) != member.password:
            print('密码错误！')
            return
        print(f'卡号 {card_num} 当前的消费积分为 {member.integral}')

def main():
    manage = Manage()
    print('欢迎使用鱼C超市会员管理系统~')
    while True:
        print('1. 会员开卡;2. 修改密码;3. 商品支付;4. 积分查询;5. 退出程序：')
        choice = int(input('请输入您的选择：'))
        if choice == 1:
            manage.new_card()
        elif choice == 2:
            manage.modify_password()
        elif choice == 3:
            manage.pay()
        elif choice == 4:
            manage.inquiry_integral()
        elif choice == 5:
            print('感谢使用鱼C超市会员管理系统~')
            break
        else:
            print('无效的选择，请重新输入。')